In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Helpers
def generate_binary(n, p): 
    return np.random.choice([0, 1], size=n, p=[1-p, p])

def generate_categorical(n, categories, probs): 
    return np.random.choice(categories, size=n, p=probs)

def generate_age(n, mu=28, sigma=7, lo=16, hi=60): 
    return np.clip(np.random.normal(loc=mu, scale=sigma, size=n).astype(int), lo, hi)

def softmax(vec, temperature=0.8):
    # Lower temperature -> more deterministic; higher -> more stochastic
    v = np.array(list(vec), dtype=float)
    v = v - v.max()
    ex = np.exp(v / max(1e-6, temperature))
    return ex / ex.sum()

# Non-IID client profiles
# - prevalence_shift: modifies feature prevalence at generation
# - score_scale: multiplies per-class scores (preference/experience)
# - missingness: optional per-feature missing rate (light)
CLIENT_PROFILES = {
    1:  {"name": "AcademicCenter", 
         "prevalence_shift": {"Impaction_Depth": [0.2,0.5,0.3], "Proximity_Nerve_p": 0.35, "Age_mu": 26},
         "score_scale": {1:0.95, 2:1.05, 3:1.05, 4:1.00},
         "missingness": {"Tooth_Mobility": 0.05}},
    2:  {"name": "HighReferralClinic", 
         "prevalence_shift": {"Impaction_Depth": [0.2,0.4,0.4], "Proximity_Nerve_p": 0.40, "Age_mu": 30},
         "score_scale": {1:0.90, 2:0.95, 3:1.05, 4:1.10},
         "missingness": {}},
    3:  {"name": "RuralGeneral", 
         "prevalence_shift": {"Impaction_Depth": [0.4,0.45,0.15], "Proximity_Nerve_p": 0.25, "Age_mu": 27},
         "score_scale": {1:1.05, 2:1.00, 3:0.95, 4:0.95},
         "missingness": {"Bone_Density": 0.07}},
    4:  {"name": "OMFS_Experienced", 
         "prevalence_shift": {"Impaction_Depth": [0.25,0.45,0.30], "Proximity_Nerve_p": 0.35, "Age_mu": 28},
         "score_scale": {1:0.95, 2:1.00, 3:1.00, 4:0.90},
         "missingness": {}},
    5:  {"name": "OlderPopulation", 
         "prevalence_shift": {"Age_mu": 35, "Proximity_Nerve_p": 0.32},
         "score_scale": {1:0.95, 2:1.00, 3:1.05, 4:1.05},
         "missingness": {}},
    # others default
}

def get_profile(client_id):
    return CLIENT_PROFILES.get(client_id, 
        {"name": f"Clinic_{client_id}",
         "prevalence_shift": {}, "score_scale": {1:1,2:1,3:1,4:1}, "missingness": {}})

# Global sizes
n_clients = 10
patients_per_client = 200

rows = []

# Generate per-client
for c in range(1, n_clients+1):
    prof = get_profile(c)
    n = patients_per_client

    # Prevalence shifts / distributions
    Age_mu = prof["prevalence_shift"].get("Age_mu", 28)
    Proximity_p = prof["prevalence_shift"].get("Proximity_Nerve_p", 0.30)
    Depth_probs = prof["prevalence_shift"].get("Impaction_Depth", [0.3,0.5,0.2])

    # Features
    age = generate_age(n, mu=Age_mu)
    sex = generate_binary(n, 0.5)

    pain = generate_binary(n, 0.5)
    swelling = generate_binary(n, 0.30)
    trismus = generate_binary(n, 0.20)
    pericoronitis = generate_binary(n, 0.25)

    caries_w = generate_binary(n, 0.20)
    caries_adj = generate_binary(n, 0.15)
    perio = generate_categorical(n, [1,2,3], [0.6,0.3,0.1])              # 1 healthy, 3 severe
    root_dev = generate_categorical(n, [1,2,3], [0.2,0.7,0.1])           # 1 incomplete, 2 complete
    mobility = generate_categorical(n, [0,1,2], [0.7,0.2,0.1])           # 0 none, 2 severe

    # Angulation: 1 vertical, 2 mesioangular, 3 distoangular, 4 horizontal
    ang = generate_categorical(n, [1,2,3,4], [0.4,0.3,0.2,0.1])

    # Depth: 1 superficial, 2 partial bony, 3 deep bony (use client-specific probs)
    depth = generate_categorical(n, [1,2,3], Depth_probs)

    prox_nerve = generate_binary(n, Proximity_p)                         # 1 = close/uncertain
    root_count = generate_categorical(n, [1,2,3,4], [0.1,0.5,0.3,0.1])
    root_curve = generate_binary(n, 0.30)                                # 1 = curved/divergent
    bone_density = generate_categorical(n, [1,2,3], [0.4,0.4,0.2])       # 3 = high

    cyst = generate_binary(n, 0.10)

    diabetes = generate_binary(n, 0.10)
    osteoporosis = generate_binary(n, 0.05)
    clotting = generate_binary(n, 0.02)
    smoking = generate_binary(n, 0.25)
    bisphosph = generate_binary(n, 0.03)
    prev_issue = generate_binary(n, 0.10)

    for i in range(n):
        rows.append({
            "Client": c, "Patient": i+1,
            "Age": age[i], "Sex": sex[i],
            "Pain": pain[i], "Swelling": swelling[i], "Trismus": trismus[i], "Pericoronitis": pericoronitis[i],
            "Caries_Wisdom": caries_w[i], "Caries_Adjacent": caries_adj[i],
            "Periodontal_Status": perio[i], "Root_Development": root_dev[i], "Tooth_Mobility": mobility[i],
            "Tooth_Angulation": ang[i], "Impaction_Depth": depth[i], "Proximity_Nerve": prox_nerve[i],
            "Root_Count": root_count[i], "Root_Curvature": root_curve[i], "Bone_Density": bone_density[i],
            "Cyst": cyst[i],
            "Diabetes": diabetes[i], "Osteoporosis": osteoporosis[i], "Clotting_Disorder": clotting[i],
            "Smoking": smoking[i], "Bisphosphonates": bisphosph[i],
            "Prev_Extraction_Issue": prev_issue[i],
        })

df = pd.DataFrame(rows)

# Optional: light missingness per profile
for c in range(1, n_clients+1):
    miss = get_profile(c)["missingness"]
    idx = df["Client"] == c
    for col, rate in miss.items():
        mask = (np.random.rand(idx.sum()) < rate)
        df.loc[idx, col] = df.loc[idx, col].mask(mask)

# Scoring: weights respecting intensity
# Class semantics:
# 1 = Simple extraction
# 2 = Surgical extraction (flap/minor bone)
# 3 = Surgical extraction (complex: deep bony/sectioning)
# 4 = Referral/Coronectomy (high IAN risk, systemic risk)
BASE_PRIOR = {1:0.4, 2:0.5, 3:0.5, 4:0.4}

def add(scores, cls, amt): 
    scores[cls] += amt

def compute_scores_row(row):
    s = {1:BASE_PRIOR[1], 2:BASE_PRIOR[2], 3:BASE_PRIOR[3], 4:BASE_PRIOR[4]}

    # ---- Symptoms / Infection (presence -> graded push to 2/3)
    if row["Pain"]==1:             add(s,2,0.45); add(s,3,0.60)
    if row["Swelling"]==1:         add(s,2,0.50); add(s,3,0.70); add(s,4,0.15)
    if row["Trismus"]==1:          add(s,2,0.55); add(s,3,0.75); add(s,4,0.20)
    if row["Pericoronitis"]==1:    add(s,2,0.45); add(s,3,0.65)

    # ---- Anatomy / Radiology (intensity-aware)
    # Depth 1/2/3 : superficial/partial/deep
    depth = row["Impaction_Depth"]
    if depth==2:                   add(s,2,0.30); add(s,3,0.40)
    elif depth==3:                 add(s,2,0.35); add(s,3,0.80); add(s,4,0.35)

    # Proximity to IAN: 1 = close/uncertain
    if row["Proximity_Nerve"]==1:  add(s,2,0.25); add(s,3,0.65); add(s,4,0.95)

    # Angulation: 1 vertical, 2 mesio, 3 disto, 4 horizontal
    ang = row["Tooth_Angulation"]
    if ang==2:                     add(s,2,0.20); add(s,3,0.25)
    elif ang==3:                   add(s,2,0.25); add(s,3,0.40)
    elif ang==4:                   add(s,2,0.25); add(s,3,0.60); add(s,4,0.20)

    # Roots
    rc = row["Root_Count"]
    if rc>=3:                      add(s,3,0.45); add(s,4,0.35)
    if row["Root_Curvature"]==1:   add(s,3,0.35); add(s,4,0.35)

    # Bone density: 3=high, harder
    if row["Bone_Density"]==3:     add(s,3,0.30); add(s,4,0.30)

    # Cyst/pathology
    if row["Cyst"]==1:             add(s,3,0.35); add(s,4,0.30)

    # Tooth mobility: 2 severe -> often easier luxation (tilts toward simple)
    mob = row["Tooth_Mobility"]
    if mob==1:                     add(s,1,0.10)
    elif mob==2:                   add(s,1,0.30); add(s,2,0.10)

    # Root development: 1 incomplete -> easier extraction; 3 resorbing -> variable, slight ↑complex
    rd = row["Root_Development"]
    if rd==1:                      add(s,1,0.25)
    elif rd==3:                    add(s,2,0.10); add(s,3,0.15)

    # Periodontal status: severe perio may facilitate extraction (less bone support) but ↑ post-op concerns
    ps = row["Periodontal_Status"]
    if ps==2:                      add(s,2,0.05)
    elif ps==3:                    add(s,1,0.15); add(s,2,0.10)

    # Caries context: drives toward surgery when symptomatic/impacted
    if row["Caries_Wisdom"]==1:    add(s,2,0.15); add(s,3,0.10)
    if row["Caries_Adjacent"]==1:  add(s,2,0.10); add(s,3,0.15)

    # ---- Systemic / risk (tilt to referral when significant)
    if row["Diabetes"]==1:         add(s,4,0.20)
    if row["Osteoporosis"]==1:     add(s,4,0.25)
    if row["Clotting_Disorder"]==1:add(s,4,0.45)
    if row["Bisphosphonates"]==1:  add(s,4,0.60)
    if row["Prev_Extraction_Issue"]==1: add(s,3,0.20); add(s,4,0.20)

    # ---- Age scaling (>=25 starts to matter; gradual)
    age = row["Age"]
    age_factor = max(0.0, (age - 25)/10.0)  # 25->0, 35->1, 45->2, etc.
    add(s,3,0.10 * min(age_factor, 2.0))
    add(s,4,0.20 * min(age_factor, 2.0))

    # ---- Interactions (clinically meaningful combos)
    # Deep + close IAN -> strong push to referral/coronectomy
    if (depth==3) and (row["Proximity_Nerve"]==1):
        add(s,4,1.00)
    # Swelling + trismus (reduced access + inflammation) -> more complex
    if (row["Swelling"]==1) and (row["Trismus"]==1):
        add(s,3,0.50)
    # Cyst + high density -> more complex dissection
    if (row["Cyst"]==1) and (row["Bone_Density"]==3):
        add(s,3,0.30)
    # Horizontal angulation + ≥3 roots -> complex/possible referral
    if (ang==4) and (rc>=3):
        add(s,3,0.40); add(s,4,0.30)

    # ---- Client preference scaling (non-IID on decision side)
    scales = get_profile(int(row["Client"]))["score_scale"]
    for k in s:
        s[k] *= scales.get(k, 1.0)

    return s

def decide_row(row, temperature=0.8, noise_sd=0.04):
    s = compute_scores_row(row)
    # Light clinician variability
    for k in s:
        s[k] += np.random.normal(0.0, noise_sd)
    # Probabilistic decision
    probs = softmax([s[1], s[2], s[3], s[4]], temperature=temperature)
    decision = np.random.choice([1,2,3,4], p=probs)
    return decision, s, probs

# Apply decisioning
decisions, score1, score2, score3, score4, p1, p2, p3, p4 = [], [], [], [], [], [], [], [], []
for i, row in df.iterrows():
    d, s, p = decide_row(row)
    decisions.append(d)
    score1.append(s[1]); score2.append(s[2]); score3.append(s[3]); score4.append(s[4])
    p1.append(p[0]); p2.append(p[1]); p3.append(p[2]); p4.append(p[3])

df["Decision"] = decisions
df["Score_1"] = score1; df["Score_2"] = score2; df["Score_3"] = score3; df["Score_4"] = score4
df["Prob_1"]  = p1;     df["Prob_2"]  = p2;     df["Prob_3"]  = p3;     df["Prob_4"]  = p4

# Save
df.to_excel("synthetic_wisdom_tooth_nonIID_realistic.xlsx", index=False)
print("✅ Saved: synthetic_wisdom_tooth_nonIID_realistic.xlsx")

✅ Saved: synthetic_wisdom_tooth_nonIID_realistic.xlsx
